# Long-Run Risk

The model of [Bansal and Yaron (2004)](https://doi.org/10.1111/j.1540-6261.2004.00670.x) with its two states, expected consumption growth `mu` and consumption variance `v`, adapted from an example of EconPDEs.jl. The second part, added in Pollis, prices real bonds in the same economy.

In [ ]:
Pollis.packages("EconPDEs", "Distributions", "Plots")

## From EconPDEs.jl

The code in this part is the package's example, with Greek letters written in Latin (`mu`, `sigma`, `kappa`, `nu`, `rho`, `gamma`, `psi`). A line changed in any other way says `# Changed in Pollis:`.

### The model

Expected growth and variance each revert to a mean:

```
d mu = kappamu (mubar - mu) dt + numu sqrt(v) dWmu
d v  = kappav (vbar - v) dt + nuv sqrt(v) dWv
```

With Epstein-Zin preferences the unknown is the wealth-consumption ratio `p(mu, v)`, which solves a nonlinear PDE. The parameters are in a struct, and the model is solved at its defaults.

In [ ]:
using EconPDEs, Distributions, Plots

Base.@kwdef struct BansalYaronModel
    mubar::Float64 = 0.018    # long-run mean of expected consumption growth
    vbar::Float64 = 0.00073  # long-run mean of consumption variance
    kappamu::Float64 = 0.252      # mean-reversion speed of expected growth
    numu::Float64 = 0.528      # volatility of expected growth
    kappav::Float64 = 0.156      # mean-reversion speed of variance
    nuv::Float64 = 0.00354    # volatility of variance (vol of vol)
    rho::Float64 = 0.024       # discount rate
    gamma::Float64 = 7.5         # relative risk aversion
    psi::Float64 = 1.5         # elasticity of intertemporal substitution
end

In [ ]:
m = BansalYaronModel()

### The grid and the guess

The grid has one key per state and spans the 1st to the 99th percentile of the stationary distributions of `mu` (normal) and `v` (gamma). The diffusion of `v` vanishes at `v = 0`, so no boundary condition is needed there. The guess is a matrix over the grid.

In [ ]:
mun, vn = 30, 30
mudistribution = Normal(m.mubar, sqrt(m.numu^2 * m.vbar / (2 * m.kappamu)))
mus = range(quantile(mudistribution, 0.01), quantile(mudistribution, 0.99), length = mun)
nudistribution = Gamma(2 * m.kappav * m.vbar / m.nuv^2, m.nuv^2 / (2 * m.kappav))
vs = range(quantile(nudistribution, 0.0), quantile(nudistribution, 0.99), length = vn)
stategrid = (; mu = mus, v = vs)

In [ ]:
guess = (; p = ones(mun, vn))

### The equation

The function follows the package's convention: it takes `state` and `u` and returns the time derivative of the unknown. Each first derivative is upwinded on the sign of its drift.

In [ ]:
function (m::BansalYaronModel)(state::NamedTuple, u::NamedTuple)
    (; mubar, vbar, kappamu, numu, kappav, nuv, rho, gamma, psi) = m
    (; mu, v) = state
    (; p, pmu_up, pmu_down, pv_up, pv_down, pmumu, pvv) = u

    # drifts and volatilities of consumption, mu and v
    muc = mu
    sigmac = sqrt(v)
    mumu = kappamu * (mubar - mu)
    sigmamu = numu * sqrt(v)
    muv = kappav * (vbar - v)
    sigmav = nuv * sqrt(v)

    # upwind each first derivative on the sign of its drift
    pmu = (mumu >= 0) ? pmu_up : pmu_down
    pv = (muv >= 0) ? pv_up : pv_down

    sigmap_Zmu = pmu / p * sigmamu
    sigmap_Zv = pv / p * sigmav
    sigmap2 = sigmap_Zmu^2 + sigmap_Zv^2
    mup = pmu / p * mumu + pv / p * muv + 0.5 * pmumu / p * sigmamu^2 + 0.5 * pvv / p * sigmav^2

    # market prices of risk
    kappa_Zc = gamma * sigmac
    kappa_Zmu = -(1 - gamma * psi) / (psi - 1) * sigmap_Zmu
    kappa_Zv = -(1 - gamma * psi) / (psi - 1) * sigmap_Zv

    r = rho + muc / psi - (1 + 1 / psi) / 2 * gamma * sigmac^2 - (gamma * psi - 1) / (2 * (psi - 1)) * sigmap2
    pt = -p * (1 / p + muc + mup - r - kappa_Zc * sigmac - kappa_Zmu * sigmap_Zmu - kappa_Zv * sigmap_Zv)
    return (; pt), (; r, kappa_Zmu, kappa_Zv, mumu, muv, sigmamu, sigmav)  # Changed in Pollis: also keeps what the bonds below need
end

### Solving it

In [ ]:
result = pdesolve(m, stategrid, guess)

### The solution

The wealth-consumption ratio rises with expected growth and falls with variance: the claim is worth more when future growth is high and less when the economy is riskier.

In [ ]:
p = result.solution.p
surface(vs, mus, p; xlabel = "variance v", ylabel = "growth mu", zlabel = "p", colorbar = false, size = (720, 460), left_margin = 8Plots.mm, bottom_margin = 8Plots.mm, right_margin = 6Plots.mm)

## Added in Pollis, not in EconPDEs.jl

Everything from here to the end was written for Pollis. It is not in the package, and the package's author did not write or review it. Each code cell starts with a comment saying so.

The example stops at the wealth-consumption ratio. The same solution also prices real bonds, since it gives the interest rate `r` and the market prices of risk `kappa_Zmu` and `kappa_Zv`. This economy has no inflation, so these yields are real, not nominal.

### A wider grid

A long bond depends on states far from the mean, beyond where the example's grid stops. So the additions solve on a wider grid, from the percentile `q` to `1 - q`, with `q = 1e-6` for the bonds. The last table tries several grids.

In [ ]:
# Added in Pollis, not from EconPDEs.jl

function solve_wide(m; n = 80, q = 1e-6)
    mudistribution = Normal(m.mubar, sqrt(m.numu^2 * m.vbar / (2 * m.kappamu)))
    mus = range(quantile(mudistribution, q), quantile(mudistribution, 1 - q), length = n)
    nudistribution = Gamma(2 * m.kappav * m.vbar / m.nuv^2, m.nuv^2 / (2 * m.kappav))
    vs = range(quantile(nudistribution, 0.0), quantile(nudistribution, 1 - q), length = n)
    stategrid = (; mu = mus, v = vs)
    result = pdesolve(m, stategrid, (; p = ones(n, n)); verbose = false)
    (; mus, vs, stategrid, result)
end

wide = solve_wide(m)
println("Converged: ", wide.result.converged)

### The term structure of real yields

A real zero-coupon bond pays one unit of consumption at maturity. Its price `B(mu, v, t)` solves a linear PDE, with drifts adjusted by the market prices of risk:

```
0 = Bt + (mumu - kappa_Zmu sigmamu) Bmu + (muv - kappa_Zv sigmav) Bv
       + 0.5 sigmamu^2 Bmumu + 0.5 sigmav^2 Bvv - r B
```

with `B = 1` at maturity. With a list of times, `pdesolve` solves backwards from the guess, which is the condition at the last time. The bond's function takes the time as a third argument, and reads the coefficients of the solved model at each grid point.

The spacing of the times matters, since `pdesolve` takes one implicit step between each pair. Steps of a month are used here. With steps of a year, the five-year price checked below came out at 0.8956 and not 0.8932.

In [ ]:
# Added in Pollis, not from EconPDEs.jl

struct RealBond{T}
    mus::Vector{Float64}
    vs::Vector{Float64}
    saved::T
end

function (b::RealBond)(state::NamedTuple, u::NamedTuple, t::Number)
    (; r, kappa_Zmu, kappa_Zv, mumu, muv, sigmamu, sigmav) = b.saved
    i, j = searchsortedfirst(b.mus, state.mu), searchsortedfirst(b.vs, state.v)
    (; B, Bmu_up, Bmu_down, Bv_up, Bv_down, Bmumu, Bvv) = u
    driftmu = mumu[i, j] - kappa_Zmu[i, j] * sigmamu[i, j]
    driftv = muv[i, j] - kappa_Zv[i, j] * sigmav[i, j]
    Bmu = driftmu >= 0 ? Bmu_up : Bmu_down
    Bv = driftv >= 0 ? Bv_up : Bv_down
    Bt = -(driftmu * Bmu + driftv * Bv + 0.5 * sigmamu[i, j]^2 * Bmumu + 0.5 * sigmav[i, j]^2 * Bvv - r[i, j] * B)
    return (; Bt)
end

maturities = vcat((360:-1:1) ./ 12, 1 / 365, 0.0)
bondresult = pdesolve(RealBond(collect(wide.mus), collect(wide.vs), wide.result.saved), wide.stategrid, (; B = ones(length(wide.mus), length(wide.vs))), maturities[1] .- maturities; verbose = false)
println("Converged: ", bondresult.converged)
yields = [-log.(bondresult.solution.B[:, :, k]) ./ maturities[k] for k in 1:(length(maturities) - 1)]

The mean curve weights each state by its stationary distribution, from a monthly simulation of `mu` and `v`. The other curves hold `v` at its mean and put `mu` two standard deviations either side of its mean.

In [ ]:
# Added in Pollis, not from EconPDEs.jl

using Printf, Random

function stationary_weights(m, mus, vs; months = 1_200_000, seed = 1)
    dt, rng = 1 / 12, Xoshiro(seed)
    weights = zeros(length(mus), length(vs))
    mu, v = m.mubar, m.vbar
    for month in 1:months
        i = clamp(searchsortedlast(mus, mu), 1, length(mus) - 1)
        j = clamp(searchsortedlast(vs, v), 1, length(vs) - 1)
        a = clamp((mu - mus[i]) / step(mus), 0, 1)
        c = clamp((v - vs[j]) / step(vs), 0, 1)
        weights[i, j] += (1 - a) * (1 - c)
        weights[i + 1, j] += a * (1 - c)
        weights[i, j + 1] += (1 - a) * c
        weights[i + 1, j + 1] += a * c
        rootv = sqrt(v)
        mu += m.kappamu * (m.mubar - mu) * dt + m.numu * rootv * sqrt(dt) * randn(rng)
        v = max(v + m.kappav * (m.vbar - v) * dt + m.nuv * rootv * sqrt(dt) * randn(rng), 0.0)
    end
    weights ./ months
end

weights = stationary_weights(m, wide.mus, wide.vs)
sdmu = sqrt(m.numu^2 * m.vbar / (2 * m.kappamu))
nearest(grid, x) = argmin(abs.(grid .- x))
jmean = nearest(wide.vs, m.vbar)
curve(i) = [100 * y[i, jmean] for y in yields]
tenors = maturities[1:(end - 1)]
plot(tenors, [sum(weights .* y) * 100 for y in yields]; label = "mean over states", color = :black, linewidth = 2, xlabel = "maturity, years", ylabel = "real yield, %", size = (620, 380))
plot!(tenors, curve(nearest(wide.mus, m.mubar + 2sdmu)); label = "mu two sd above its mean")
plot!(tenors, curve(nearest(wide.mus, m.mubar - 2sdmu)); label = "mu two sd below its mean")

The mean curve slopes down, the same direction as the negative yield spread [Beeler and Campbell (2012)](https://doi.org/10.1561/104.00000004) report for a real perpetuity under this calibration. The values at `gamma = 7.5` are not compared with any paper: Beeler and Campbell tabulate `gamma` at 5, 10 and 15, and only for the perpetuity below.

Widening the grid to 160 x 160 with `q = 1e-10` changed the mean yields at 3 months and at 1, 5, 10 and 30 years by 0.013 of a percentage point or less.

### Two checks on the bonds

A very short yield should approach `r`. And the five-year price should agree with a Monte Carlo average of `exp(-integral of r)` over paths simulated with the same adjusted drifts, a calculation that does not use the bond PDE. Its standard error is printed with it.

In [ ]:
# Added in Pollis, not from EconPDEs.jl

using Statistics

function montecarlo_price(m, wide, i, j; maturity = 5.0, paths = 20_000, seed = 2)
    (; mus, vs) = wide
    (; r, kappa_Zmu, kappa_Zv, mumu, muv, sigmamu, sigmav) = wide.result.saved
    dt, rng = 1 / 48, Xoshiro(seed)
    function bilinear(f, mu, v)
        a0 = clamp(searchsortedlast(mus, mu), 1, length(mus) - 1)
        c0 = clamp(searchsortedlast(vs, v), 1, length(vs) - 1)
        a, c = (mu - mus[a0]) / step(mus), (v - vs[c0]) / step(vs)
        (1 - a) * (1 - c) * f[a0, c0] + a * (1 - c) * f[a0 + 1, c0] + (1 - a) * c * f[a0, c0 + 1] + a * c * f[a0 + 1, c0 + 1]
    end
    driftmu = mumu .- kappa_Zmu .* sigmamu
    driftv = muv .- kappa_Zv .* sigmav
    prices = zeros(paths)
    for path in 1:paths
        mu, v, integral = mus[i], vs[j], 0.0
        for _ in 1:round(Int, maturity / dt)
            integral += bilinear(r, mu, v) * dt
            rootv = sqrt(v)
            mu1 = mu + bilinear(driftmu, mu, v) * dt + m.numu * rootv * sqrt(dt) * randn(rng)
            v1 = v + bilinear(driftv, mu, v) * dt + m.nuv * rootv * sqrt(dt) * randn(rng)
            mu, v = clamp(mu1, mus[1], mus[end]), clamp(v1, vs[1], vs[end])
        end
        prices[path] = exp(-integral)
    end
    mean(prices), std(prices) / sqrt(paths)
end

imean = nearest(wide.mus, m.mubar)
println("r at the mean state: ", round(100 * wide.result.saved.r[imean, jmean], digits = 3), "%")
println("One-day yield at the mean state: ", round(100 * yields[end][imean, jmean], digits = 3), "%")
k5 = findfirst(==(5.0), maturities)
println("Five-year price, bond PDE: ", round(bondresult.solution.B[imean, jmean, k5], digits = 4))
price, standarderror = montecarlo_price(m, wide, imean, jmean)
println("Five-year price, Monte Carlo: ", round(price, digits = 4), " (standard error ", round(standarderror, digits = 4), ")")

### The real perpetuity, against a published table

The working-paper version of [Beeler and Campbell (2012)](https://doi.org/10.1561/104.00000004) reports a real perpetuity, which pays one unit of consumption a year for ever, with the same parameters. Its Table IX, Panel A has the mean yield `E(y_b)` and the mean yield spread `E(y_b - r^f)`, at `psi = 1.5` and `gamma` of 5 and 10, from 1.2 million simulated months. Here the price `Pb` solves the stationary version of the bond's equation, with a payment of one a year, and the yield is `1 / Pb`.

In [ ]:
# Added in Pollis, not from EconPDEs.jl

struct RealPerpetuity{T}
    mus::Vector{Float64}
    vs::Vector{Float64}
    saved::T
end

function (b::RealPerpetuity)(state::NamedTuple, u::NamedTuple)
    (; r, kappa_Zmu, kappa_Zv, mumu, muv, sigmamu, sigmav) = b.saved
    i, j = searchsortedfirst(b.mus, state.mu), searchsortedfirst(b.vs, state.v)
    (; Pb, Pbmu_up, Pbmu_down, Pbv_up, Pbv_down, Pbmumu, Pbvv) = u
    driftmu = mumu[i, j] - kappa_Zmu[i, j] * sigmamu[i, j]
    driftv = muv[i, j] - kappa_Zv[i, j] * sigmav[i, j]
    Pbmu = driftmu >= 0 ? Pbmu_up : Pbmu_down
    Pbv = driftv >= 0 ? Pbv_up : Pbv_down
    Pbt = -(1 + driftmu * Pbmu + driftv * Pbv + 0.5 * sigmamu[i, j]^2 * Pbmumu + 0.5 * sigmav[i, j]^2 * Pbvv - r[i, j] * Pb)
    return (; Pbt)
end

function perpetuity_moments(gamma; n, q)
    model = BansalYaronModel(; gamma)
    sol = solve_wide(model; n, q)
    perpetuity = pdesolve(RealPerpetuity(collect(sol.mus), collect(sol.vs), sol.result.saved), sol.stategrid, (; Pb = fill(40.0, n, n)); verbose = false)
    @assert sol.result.converged && perpetuity.converged
    weights = stationary_weights(model, sol.mus, sol.vs)
    yield, rf = sum(weights .* (1 ./ perpetuity.solution.Pb)), sum(weights .* sol.result.saved.r)
    (; yield = 100 * yield, spread = 100 * (yield - rf), rf = 100 * rf)
end

grids = (("30 x 30, q = 0.01 (example's)", 30, 0.01), ("80 x 80, q = 1e-6", 80, 1e-6), ("160 x 160, q = 1e-10", 160, 1e-10))
println(rpad("gamma", 7), rpad("", 31), rpad("E(y_b)", 9), rpad("E(y_b - r^f)", 15), "E(r^f)")
for (gamma, publishedyield, publishedspread) in ((5.0, "2.25", "-0.86"), (10.0, "0.45", "-2.14"))
    for (label, n, q) in grids
        a = perpetuity_moments(gamma; n, q)
        println(rpad(gamma, 7), rpad(label, 31), rpad(@sprintf("%.2f", a.yield), 9), rpad(@sprintf("%.2f", a.spread), 15), @sprintf("%.2f", a.rf))
    end
    println(rpad(gamma, 7), rpad("Beeler and Campbell, Table IX", 31), rpad(publishedyield, 9), publishedspread)
end

The published values are from the [NBER Working Paper 14788](https://www.nber.org/papers/w14788) version of Beeler and Campbell's paper, Table IX, Panel A; its Table II gives `E(r^f)` of 2.56 at `gamma = 10`. The published article of 2012 leaves this table out. The replication *Long-run risks: the model and aggregate asset prices*, in **Compose > Replication > Browse Replications...**, recomputes the discrete monthly model of the article's tables, and Table IX of the working paper.

The grid matters. On the example's grid the yields are well above the table, since a perpetuity's yield depends on states far in the tails.

At `gamma = 5` the two wider grids differ by at most 0.01. Their yield is 0.03 from the published one and their spread 0.01 to 0.02 from it.

At `gamma = 10` the yield keeps falling as the grid widens. So the closeness of the 80 x 80 grid to 0.45 proves nothing by itself, since a wider grid moves away from it, and the published spread lies between the values of the two wider grids. The difference that remains comes from the model, not the grid. The table simulates the discrete monthly model, where the shock to the variance is Gaussian, and this is a continuous-time solution where it is scaled by `sqrt(v)`. From the discrete model to the PDE at `gamma = 10`: log-linear consol 0.448; exact sum of the zero-coupon prices 0.445; the shock scaled by `sqrt(v)` 0.391; the example's annual parameters in the continuous limit 0.386; the PDE, with the exact wealth ratio, 0.365. At `gamma = 5` the same steps give 2.255, 2.250, 2.249, 2.244 and 2.217.

The table has no values at `gamma = 15`, where the price of the consol bond is infinite. That row is left out here because on a bounded grid the solver would still return a finite number.

## Source and Licence

The code under **From EconPDEs.jl** is adapted from `examples/asset_pricing/bansal_yaron.jl` of [EconPDEs.jl](https://github.com/matthieugomez/EconPDEs.jl), version 3.1.0, by Matthieu Gomez. Changes made in Pollis: Greek letters written in Latin; the model function also returns `r`, `kappa_Zmu`, `kappa_Zv`, `mumu`, `muv`, `sigmamu` and `sigmav`.

Added in Pollis, not in the package: the wider grid, the real bonds and their two checks, the real perpetuity and its comparison with Beeler and Campbell.

The notice below covers the package's code only, not the parts added in Pollis.

The EconPDEs.jl package is licensed under the MIT "Expat" License:

> Copyright (c) 2015: Matthieu Gomez.
>
> Permission is hereby granted, free of charge, to any person obtaining
> a copy of this software and associated documentation files (the
> "Software"), to deal in the Software without restriction, including
> without limitation the rights to use, copy, modify, merge, publish,
> distribute, sublicense, and/or sell copies of the Software, and to
> permit persons to whom the Software is furnished to do so, subject to
> the following conditions:
>
> The above copyright notice and this permission notice shall be
> included in all copies or substantial portions of the Software.
>
> THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
> EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
> MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
> IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY
> CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,
> TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
> SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

## References

Bansal, R. and Yaron, A. (2004). Risks for the long run: a potential resolution of asset pricing puzzles. *The Journal of Finance* 59(4), 1481–1509. [doi:10.1111/j.1540-6261.2004.00670.x](https://doi.org/10.1111/j.1540-6261.2004.00670.x)

Beeler, J. and Campbell, J. Y. (2012). The long-run risks model and aggregate asset prices: an empirical assessment. *Critical Finance Review* 1(1), 141–182. [doi:10.1561/104.00000004](https://doi.org/10.1561/104.00000004)